In [2]:
import pandas as pd

run_id = "<run_id>"
df = pd.read_parquet(f"../data/processed/events/AttackerType1-start=3,0.1-#0.parquet")

df.head()

,run_id,receiver_id,sender_id,receiver_is_attacker,sender_is_attacker,rcvTime,sendTime,messageID,RSSI,pos_x,...,snd_pos_x_true,snd_pos_y_true,snd_spd_x_true,snd_spd_y_true,err_pos,err_spd,attack_msg,eps_pos,eps_spd,missing_sender_truth_msgs
0,"AttackerType1-start=3,0.1-#0",7,13,0,0,10800.392893,10800.392785,38,1.115287e-07,3597.152086,...,3596.835205,5546.067452,-3.168814,38.682367,3.881189,0.116993,0,4.035205,1.02665,0
1,"AttackerType1-start=3,0.1-#0",7,31,0,0,10800.456258,10800.456150,104,3.218740e-09,3596.805861,...,3596.934491,5689.118100,1.286291,-29.953051,2.998077,0.089131,0,4.035205,1.02665,0
2,"AttackerType1-start=3,0.1-#0",7,25,0,0,10800.582131,10800.582020,137,2.900248e-09,3597.770493,...,3597.768478,5761.346297,-0.020170,-34.583899,3.454597,0.015051,0,4.035205,1.02665,0
3,"AttackerType1-start=3,0.1-#0",7,43,0,0,10800.670748,10800.670638,172,1.746131e-08,3822.892006,...,3823.045106,5266.690045,1.519114,-1.150633,0.192061,0.268147,0,4.035205,1.02665,0
4,"AttackerType1-start=3,0.1-#0",7,19,0,0,10800.870128,10800.870012,193,3.710398e-07,3617.100034,...,3617.595048,5413.785182,4.944702,-39.932099,4.028131,0.054937,0,4.035205,1.02665,0


In [3]:
df["attack_msg"].mean()          # attack比例
df.groupby("sender_is_attacker")["attack_msg"].mean()

sender_is_attacker
0    0.000509
1    1.000000
Name: attack_msg, dtype: float64

In [6]:
print(len(df))

4004


In [7]:
df["err_pos"].describe()

count    4004.000000
mean       40.302228
std       272.417910
min         0.000000
25%         0.000000
50%         2.993067
75%         3.032991
max      2005.606497
Name: err_pos, dtype: float64

## Full phase1_binary: val/test JSONL 条数统计与 split 一致性

- 统计完整版 `phase1_binary` 的 `val.jsonl` 和 `test.jsonl` 各有多少条（每行一个窗口样本）。
- 从每条 JSONL 的 `input.run_id` 收集 run_id，与 `split_v1.json` 中的 val/test run 列表对照，确认是否一致。

In [1]:
import json
from pathlib import Path

# 路径（相对 notebook 所在目录为 notebooks/）
repo_root = Path("..")
split_path = repo_root / "data/processed/splits/split_v1.json"
jsonl_dir = repo_root / "data/processed/jsonl/phase1_binary"
val_path = jsonl_dir / "val.jsonl"
test_path = jsonl_dir / "test.jsonl"

split = json.loads(split_path.read_text(encoding="utf-8"))
runs = split["runs"]
expected_val_runs = set(runs["val"])
expected_test_runs = set(runs["test"])
print("Split 文件:", split_path)
print("版本:", split.get("version", "?"))
print("Split 中 run 数量:", split.get("counts", {}))
print("Val run 数:", len(expected_val_runs))
print("Test run 数:", len(expected_test_runs))
print("Val / Test 是否有交集:", bool(expected_val_runs & expected_test_runs))

Split 文件: ..\data\processed\splits\split_v1.json
版本: per_config_4train_1pool_type_stratified
Split 中 run 数量: {'train': 180, 'val': 22, 'test': 23}
Val run 数: 22
Test run 数: 23
Val / Test 是否有交集: False


In [2]:
def count_lines_and_run_ids(path: Path) -> tuple[int, set[str]]:
    """流式读取 JSONL：统计行数并收集所有 run_id（来自 input.run_id）。"""
    n_lines = 0
    run_ids = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            n_lines += 1
            if n_lines % 1_000_000 == 0:
                print(f"  {path.name}: 已读 {n_lines:,} 行 ...")
            try:
                obj = json.loads(line)
                rid = obj.get("input", {}).get("run_id")
                if rid is not None:
                    run_ids.add(rid)
            except (json.JSONDecodeError, KeyError) as e:
                # 只记一次警告，避免刷屏
                if n_lines <= 1:
                    print(f"  解析警告 (首行): {e}")
    return n_lines, run_ids

print("Val JSONL:", val_path)
print("存在:", val_path.exists())
if val_path.exists():
    n_val, val_run_ids = count_lines_and_run_ids(val_path)
    print(f"Val 总行数: {n_val:,}")
    print(f"Val 中出现的 run_id 数: {len(val_run_ids)}")
else:
    n_val, val_run_ids = 0, set()

print()
print("Test JSONL:", test_path)
print("存在:", test_path.exists())
if test_path.exists():
    n_test, test_run_ids = count_lines_and_run_ids(test_path)
    print(f"Test 总行数: {n_test:,}")
    print(f"Test 中出现的 run_id 数: {len(test_run_ids)}")
else:
    n_test, test_run_ids = 0, set()

Val JSONL: ..\data\processed\jsonl\phase1_binary\val.jsonl
存在: True
  val.jsonl: 已读 1,000,000 行 ...
Val 总行数: 1,443,934
Val 中出现的 run_id 数: 22

Test JSONL: ..\data\processed\jsonl\phase1_binary\test.jsonl
存在: True
  test.jsonl: 已读 1,000,000 行 ...
  test.jsonl: 已读 2,000,000 行 ...
Test 总行数: 2,771,325
Test 中出现的 run_id 数: 23


In [3]:
# 与 split_v1.json 对照
print("=== 与 split 文件对照 ===\n")

# Val
print("Val:")
print(f"  Split 中 val run 数: {len(expected_val_runs)}")
print(f"  JSONL 中出现的 run_id 数: {len(val_run_ids)}")
missing_in_val = expected_val_runs - val_run_ids
extra_in_val = val_run_ids - expected_val_runs
if not missing_in_val and not extra_in_val:
    print("  ✓ Val JSONL 的 run_id 与 split 完全一致")
else:
    if missing_in_val:
        print(f"  ✗ 在 split 中但未在 val.jsonl 出现: {len(missing_in_val)} 个", list(missing_in_val)[:5], "..." if len(missing_in_val) > 5 else "")
    if extra_in_val:
        print(f"  ✗ 在 val.jsonl 出现但不在 split val 中: {len(extra_in_val)} 个", list(extra_in_val)[:5], "..." if len(extra_in_val) > 5 else "")

print()
print("Test:")
print(f"  Split 中 test run 数: {len(expected_test_runs)}")
print(f"  JSONL 中出现的 run_id 数: {len(test_run_ids)}")
missing_in_test = expected_test_runs - test_run_ids
extra_in_test = test_run_ids - expected_test_runs
if not missing_in_test and not extra_in_test:
    print("  ✓ Test JSONL 的 run_id 与 split 完全一致")
else:
    if missing_in_test:
        print(f"  ✗ 在 split 中但未在 test.jsonl 出现: {len(missing_in_test)} 个", list(missing_in_test)[:5], "..." if len(missing_in_test) > 5 else "")
    if extra_in_test:
        print(f"  ✗ 在 test.jsonl 出现但不在 split test 中: {len(extra_in_test)} 个", list(extra_in_test)[:5], "..." if len(extra_in_test) > 5 else "")

print()
print("汇总:")
print(f"  val.jsonl  总条数: {n_val:,}")
print(f"  test.jsonl 总条数: {n_test:,}")
print(f"  test/val 条数比:   {n_test / n_val:.2f}" if n_val else "  (val 为空)")

=== 与 split 文件对照 ===

Val:
  Split 中 val run 数: 22
  JSONL 中出现的 run_id 数: 22
  ✓ Val JSONL 的 run_id 与 split 完全一致

Test:
  Split 中 test run 数: 23
  JSONL 中出现的 run_id 数: 23
  ✓ Test JSONL 的 run_id 与 split 完全一致

汇总:
  val.jsonl  总条数: 1,443,934
  test.jsonl 总条数: 2,771,325
  test/val 条数比:   1.92
